In [ ]:
## General imports
from multiprocessing import Manager, Pool, cpu_count
from functools import partial
## Local imports
import get_data_matrix as getMat
import mapLearningUtils as mlu
import train_MappingJM as trMap

In [ ]:
## Pre-processing parameters
savePath      = './all_params/folded_ablated_data/'
ablationsList = ['velocity', 'anthropo', 'forcestorques', 'forces', 'torques', 'Fx', 'Fy', 'Fz', 'Tx', 'Ty', 'Tz']
foldsList     = ['2', '5']

## Mapping parameters
# Common
dataDir         = './all_params/folded_ablated_data/'
fittedModelsDir = './all_params/all_fitted_models/'
fitTypesList    = ['MVLR', 'MVPR'] # FNO, LSTM

# MVPR
degreesList = [2, 3] # Degree of the fitted polynomial for each input

# FNO
nbModesList   = [4, 8, 16, 32]       # Number of modes [number of low Fourier frequencies]
nbHChanList   = [8, 16, 32, 64] # Number of hidden channels [dimensionnality of the network's internal features]
nbMaxIterList = [100, 250, 500]      # Maximum number of training iterations
limLoss       = 1e-6                 # Convergence limit (if less improvement between two iterations then training stops)

# LSTM
# TODO

In [ ]:
## Prepare lists of parameters dictionnaries to train different versions of the models
list_params_sets = []
for fitting in fitTypesList:
    for ablation in ablationsList:
        for fold in foldsList:
            fold_name = fold + '-fold'
            data_path = dataDir + ablation + '_' + fold_name + '.pkl'
            if fitting == 'MVLR':
                saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '.pkl'
                dict_params = {'model': fitting, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                               'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                list_params_sets.append(dict_params)
            elif fitting == 'MVPR':
                for degree in degreesList:
                    saveDirFile = fittedModelsDir + fitting + '_' + ablation + '_' + fold_name + '_deg' + str(degree) + '.pkl'
                    dict_params = {'model': fitting, 'degree': degree, 'ablation': ablation, 'fold': fold_name, 'nb_folds': int(fold),
                                   'dataPath': data_path, 'fittedModelDir': fittedModelsDir, 'saveFittedDir': saveDirFile}
                    list_params_sets.append(dict_params)
            elif fitting == 'FNO':
                "FNO is not yet handled: Needs HPC"
            else:
                "The requested fitting type is not yet handled."

In [2]:
import pickle
test_path = './all_params/folded_ablated_data/anthropo_2-fold.pkl'

with open(test_path, 'rb') as file:
    data = pickle.load(file)

data.get('fold1')

{'train_subjs_calib': ['S3', 'S5', 'S17', 'S15', 'S7', 'S12', 'S14', 'S9'],
 'eval_subjs_calib': array(['S6', 'S11', 'S10', 'S16', 'S1', 'S4', 'S8', 'S2', 'S13'],
       dtype=object),
 'train_subjs_assist': ['S02', 'S12', 'S01', 'S09', 'S06'],
 'eval_subjs_assist': array(['S03', 'S10', 'S04', 'S05', 'S14'], dtype=object),
 'train_data': array([[-0.7528415843652088, 0.2204495347441032, 6.461225842779359, ...,
         -6.107226230014483e-05, 0, 'S3'],
        [-0.748226702725073, 0.2270146473467504, 5.7456410942025, ...,
         0.00015943775502084783, 0, 'S3'],
        [-0.743481680265818, 0.23358364375536692, 5.874946345356768, ...,
         0.00037661809183328886, 0, 'S3'],
        ...,
        [-0.23298110785254705, 0.33139516757703047, 8.092286527288826,
         ..., -0.005213438570570988, 25, 'S12'],
        [-0.22710567961882, 0.3322390441417388, 6.420434782567056, ...,
         -0.005392891374623072, 25, 'S12'],
        [-0.22208365179928535, 0.3334333348173328, 5.28299917741